In [0]:
%run /Users/calvin.waldheim@gmail.com/lakebase_config

In [0]:
# Cell 1 - Install
%pip install psycopg2-binary

In [0]:
# Cell 2 - Retrieval function
import psycopg2
import json
from mlflow.deployments import get_deploy_client


client = get_deploy_client("databricks")

def embed(text):
    response = client.predict(
        endpoint="databricks-gte-large-en",
        inputs={"input": [text]}
    )
    return response["data"][0]["embedding"]

def retrieve(query, project_id="memory-kb-poc", top_k=3):
    query_embedding = embed(query)
    
    conn = psycopg2.connect(CONN_STRING, password=TOKEN)
    cur = conn.cursor()
    
    cur.execute("""
        SELECT rule, context, quality_score,
               embedding <=> %s::vector AS distance
        FROM memories
        WHERE project_id = %s
        ORDER BY distance ASC
        LIMIT %s
    """, (json.dumps(query_embedding), project_id, top_k))
    
    results = cur.fetchall()
    cur.close()
    conn.close()
    return results

# Test it
query = "How does memory scaling reduce reasoning steps?"
results = retrieve(query)

for i, (rule, context, score, distance) in enumerate(results):
    print(f"\n--- Result {i+1} (distance: {distance:.3f}) ---")
    print(context[:300])

In [0]:
for query in [
    "What is the difference between episodic and semantic memory?",
    "How does governance work across projects?",
    "What happens during the distillation pipeline?"
]:
    print(f"\nQUERY: {query}")
    results = retrieve(query)
    print(f"Top result (distance: {results[0][3]:.3f}):")
    print(results[0][1][:200])